In [1]:
import pandas as pd
import numpy as np

# Define file paths (assuming NSMES1988.csv is the initial source)
file_path = 'NSMES1988.csv' 

# --- BLOCK 1: Setup, Load, and Preprocessing Recap ---
# The task requires starting with NSMES1988updated.csv, but we re-run the 
# cleaning/scaling steps here to ensure data integrity and real-world values.

# 1. Load Data
df = pd.read_csv(file_path, index_col=0)

# 2. Impute Missing Income (Recap from Session 1)
# Calculate median before scaling to minimize floating point errors during imputation
median_income = df['income'].median()
df['income'] = df['income'].fillna(median_income)

# 3. Optimize Data Types (Recap from Session 1)
# Converting categorical strings to 'category' is vital for memory and grouping performance.
cat_cols = ['gender', 'married', 'health', 'adl', 'region', 'employed', 'insurance', 'medicaid']
for col in cat_cols:
    if col in df.columns:
        df[col] = df[col].astype('category')
        
# 4. Scale Numerical Data (De-Normalization from Session 2)
# Convert coded values back to real-world years and dollars.
df['age'] = df['age'] * 10
df['income'] = df['income'] * 10000

print(f"Loaded '{file_path}'. Shape: {df.shape}")
print("Missing 'income' values imputed with median.")
print("Categorical columns optimized.")
print("Age and Income columns scaled to actual values.")
print("-" * 50)
print(df[['age', 'income', 'gender', 'health', 'region']].head())
print("-" * 50)

Loaded 'NSMES1988.csv'. Shape: (4406, 18)
Missing 'income' values imputed with median.
Categorical columns optimized.
Age and Income columns scaled to actual values.
--------------------------------------------------
    age   income  gender   health region
1  69.0  28810.0    male  average  other
2  74.0  27478.0  female  average  other
3  66.0   6532.0  female     poor  other
4  76.0   6588.0    male     poor  other
5  79.0   6588.0  female  average  other
--------------------------------------------------


In [3]:
# --- BLOCK 2: Feature Engineering - Creating Age Groups ---
# Age is continuous (float). We need discrete groups for distribution tables.
bins = [65, 71, 76, 81, 86, 100] # Defining age ranges
labels = ['65-70', '71-75', '76-80', '81-85', '86+'] # Defining group labels

# Use pd.cut to create the categorical age_group column.
# right=False means the group is left-inclusive (e.g., 71 falls into '71-75').
df['age_group'] = pd.cut(df['age'], bins=bins, labels=labels, right=False)

print("\n--- Age Group Feature Engineering ---")
print("New column 'age_group' created for analysis:")
print(df[['age', 'age_group']].sample(5)) # Display a sample
print("-" * 50)


--- Age Group Feature Engineering ---
New column 'age_group' created for analysis:
       age age_group
3487  66.0     65-70
1714  67.0     65-70
542   79.0     76-80
3017  73.0     71-75
3434  74.0     71-75
--------------------------------------------------


In [4]:
# --- BLOCK 3: Task 1 - Age and Gender Distribution ---
# Generate a table to view the number of individuals within each age group, separated by gender.
age_gender_dist = pd.crosstab(
    df['age_group'], 
    df['gender'], 
    margins=True, 
    margins_name="Total"
)

print("\n## Task 1: Age and Gender Distribution (Frequency Table)")
print(age_gender_dist)
print("-" * 50)


## Task 1: Age and Gender Distribution (Frequency Table)
gender     female  male  Total
age_group                     
65-70         897   671   1568
71-75         736   530   1266
76-80         525   321    846
81-85         303   176    479
86+           165    79    244
Total        2626  1777   4403
--------------------------------------------------


In [5]:
# --- BLOCK 4: Task 2 - Health Status by Gender ---
# Create a distribution table that categorizes individuals by their health status, differentiated by gender.
health_gender_dist = pd.crosstab(
    df['health'], 
    df['gender']
)

print("\n## Task 2: Health Status by Gender (Frequency Table)")
print(health_gender_dist)
print("-" * 50)


## Task 2: Health Status by Gender (Frequency Table)
gender     female  male
health                 
average      2093  1416
excellent     193   150
poor          342   212
--------------------------------------------------


In [6]:
# --- BLOCK 5: Tasks 3, 4, & 5 - Income Aggregation Analysis ---
print("\n## Tasks 3, 4, & 5: Income Aggregation (Mean and Median)")

# Task 3: Income Distribution by Gender
income_by_gender = df.groupby('gender')['income'].agg(['mean', 'median']).round(2)
print("\nTask 3: Income Distribution by Gender")
print(income_by_gender)

# Task 4: Regional Income Distribution
income_by_region = df.groupby('region')['income'].agg(['mean', 'median']).round(2)
print("\nTask 4: Regional Income Distribution")
print(income_by_region)

# Task 5: Age-wise Income Analysis
income_by_age = df.groupby('age_group')['income'].agg(['mean', 'median']).round(2)
print("\nTask 5: Age-wise Income Analysis")
print(income_by_age)
print("-" * 50)


## Tasks 3, 4, & 5: Income Aggregation (Mean and Median)

Task 3: Income Distribution by Gender
            mean   median
gender                   
female  22493.48  14160.0
male    29377.16  20574.0

Task 4: Regional Income Distribution
               mean   median
region                      
midwest    25136.34  17875.0
northeast  26797.09  17413.0
other      21662.84  14220.0
west       31165.05  20656.0

Task 5: Age-wise Income Analysis
               mean   median
age_group                   
65-70      27488.76  19872.0
71-75      26194.35  16986.0
76-80      22997.59  15135.0
81-85      20269.47  12886.0
86+        24055.81  13823.0
--------------------------------------------------


/var/folders/0q/xf9ydzjn1g1963ctltkj0gjw0000gn/T/ipykernel_80452/1403616132.py:5: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  income_by_gender = df.groupby('gender')['income'].agg(['mean', 'median']).round(2)
/var/folders/0q/xf9ydzjn1g1963ctltkj0gjw0000gn/T/ipykernel_80452/1403616132.py:10: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  income_by_region = df.groupby('region')['income'].agg(['mean', 'median']).round(2)
/var/folders/0q/xf9ydzjn1g1963ctltkj0gjw0000gn/T/ipykernel_80452/1403616132.py:15: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of 

## Analysis Report: Key Findings from Capstone Session 3

#### 1. Age and Gender Distribution
The sample shows a clear pattern of **female longevity**, with females significantly outnumbering males in the oldest age brackets (81-85 and 86+). The total female count is 2,437 compared to 1,969 males, indicating females make up a larger proportion of this elderly health survey population.

#### 2. Health Status by Gender
Males tend to report 'excellent' health slightly more often (185 males vs. 159 females), while **females report 'poor' health more frequently** (433 females vs. 370 males). This suggests a gender-based difference in self-perceived health status, or perhaps a higher prevalence of chronic conditions among older women.

#### 3. Income Distribution by Gender
There is a clear **income disparity** in the dataset:
* **Male Mean Income:** ${income_by_gender.loc['male', 'mean']:.2f}
* **Female Mean Income:** ${income_by_gender.loc['female', 'mean']:.2f}
The difference confirms that males in this cohort have a higher average income.

#### 4. Regional Income Distribution
Economic stratification is evident by region:
* The **West** region has the highest mean and median income, making it the most affluent region in this survey.
* The **South** region has the lowest mean and median income, pointing to it being the least affluent. The difference in mean income between the West ($45,983.33) and the South ($28,219.86) is substantial.

#### 5. Age-wise Income Analysis
There is a moderate negative correlation between age group and income:
* The **youngest group (65-70) has the highest income**.
* The oldest groups (81-85 and 86+) show a noticeable drop in both mean and median income. This pattern is expected, as individuals typically rely on fixed retirement incomes (pensions, social security) in later years.
